# 08. Local LLM Inference and Evaluation

This notebook evaluates compact instruction-tuned Large Language Models running locally via `llama.cpp`:
- **Meta Llama 3.2 1B Instruct** (`Q4_K_M`)
- **Qwen 2.5 1.5B Instruct** (`Q4_K_M`)

### Objectives:
1. Serve local GGUF models on GPU via `llama-server.exe`.
2. Supply identical analytical evidence (XGBoost metrics and SHAP feature attributions).
3. Generate grounded natural-language explanations.
4. Quantify performance: latency, throughput (tokens/sec), factual consistency, and numerical accuracy.

In [ ]:
import os
import sys
import time
import json
import pandas as pd
import numpy as np

# Add project root to sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.llm import LocalLLMManager
from src.explainability import load_shap_final_summary, load_sample_explanation
from src.evaluation import evaluate_response_quality

print("Modules imported successfully.")

In [ ]:
# Validate model files and runtime existence
manager = LocalLLMManager()

llama_model = manager.get_model_path("llama")
qwen_model = manager.get_model_path("qwen")
server_exe = manager.get_server_executable()

print(f"Llama model: {llama_model.name} ({round(os.path.getsize(llama_model)/(1024**3), 2)} GB)")
print(f"Qwen model:  {qwen_model.name} ({round(os.path.getsize(qwen_model)/(1024**3), 2)} GB)")
print(f"Server executable: {server_exe.name} (Ready: {server_exe.exists()})")

In [ ]:
# Load analytical evidence for prompt construction
shap_summary = load_shap_final_summary().head(5)
sample_explanation = load_sample_explanation(0).head(5)

print("Top 5 SHAP Feature Drivers:")
display(shap_summary)

print("\nSample 0 Customer Feature Contributions:")
display(sample_explanation)

In [ ]:
# Load prompt templates
prompts_dir = os.path.join(project_root, "prompts")

with open(os.path.join(prompts_dir, "analysis.txt"), "r", encoding="utf-8") as f:
    analysis_template = f.read()

with open(os.path.join(prompts_dir, "comparison.txt"), "r", encoding="utf-8") as f:
    comparison_prompt = f.read()

features_str = "\n".join([
    f"- {row['Feature']}: Absolute SHAP impact = {row['Mean_Absolute_SHAP']:.2f} points ({row['Direction']} impact on score)"
    for _, row in shap_summary.iterrows()
])

highlights_str = (
    "- Age: 33 | Gender: Male | Occupation: Self-Employed | Card Type: Silver\n"
    "- Payment Ratio: 0.46 | Credit Utilization: 0.70\n"
    "- Outstanding Balance: 154,000 INR | Monthly Spending: 36,733 INR"
)

customer_prompt = analysis_template.format(
    predicted_score="639.6",
    actual_score="648.0",
    top_features_summary=features_str,
    customer_highlights=highlights_str
)

print("Prompts prepared successfully.")

In [ ]:
# Run Llama 3.2 1B Evaluation
print("Starting Llama 3.2 1B server...")
manager.start_server("llama")

try:
    llama_res = manager.generate(customer_prompt, max_tokens=300, temperature=0.2)
    print(f"Llama Generation Completed in {llama_res['latency_seconds']}s ({llama_res['tokens_per_second']} t/s)\n")
    print("--- LLAMA EXPLANATION ---")
    print(llama_res["content"])
    
    llama_eval = evaluate_response_quality(
        llama_res["content"],
        expected_numbers=[639.6, 648.0],
        top_features=["Payment_Ratio", "Credit_Utilization"]
    )
    print(f"\nLlama Quality Score: {llama_eval['composite_quality_score']} (Rating: {llama_eval['rating']})")
finally:
    manager.stop_server()
    print("Llama server stopped.")

In [ ]:
# Run Qwen 2.5 1.5B Evaluation
print("Starting Qwen 2.5 1.5B server...")
manager.start_server("qwen")

try:
    qwen_res = manager.generate(customer_prompt, max_tokens=300, temperature=0.2)
    print(f"Qwen Generation Completed in {qwen_res['latency_seconds']}s ({qwen_res['tokens_per_second']} t/s)\n")
    print("--- QWEN EXPLANATION ---")
    print(qwen_res["content"])
    
    qwen_eval = evaluate_response_quality(
        qwen_res["content"],
        expected_numbers=[639.6, 648.0],
        top_features=["Payment_Ratio", "Credit_Utilization"]
    )
    print(f"\nQwen Quality Score: {qwen_eval['composite_quality_score']} (Rating: {qwen_eval['rating']})")
finally:
    manager.stop_server()
    print("Qwen server stopped.")

In [ ]:
# Load and display comparative benchmark metrics
benchmark_csv = os.path.join(project_root, "results", "metrics", "llm_benchmark_results.csv")
if os.path.exists(benchmark_csv):
    df_bench = pd.read_csv(benchmark_csv)
    print("=== LLM Comparative Benchmark Metrics ===")
    display(df_bench)
else:
    print("Run benchmark pipeline to populate results.")